# 04. 도구와 구조화된 출력

LangChain v1에서 `@tool` 데코레이터로 커스텀 도구를 만들고, `with_structured_output()`으로 구조화된 응답을 받는 방법을 학습합니다.

도구가 필요한가?  
│  
├─ 아니오 (LLM 1번이면 충분)  
│   └─ → with_structured_output(Schema)        ← 가장 가벼움  
│  
└─ 예  
       │  
       ├─ 답을 구조화할 필요 없음  
       │   └─ → create_agent(model, tools, system_prompt)  
       │  
       └─ 답도 구조화 필요  
           ├─ 모델이 네이티브 JSON 모드 지원? → ProviderStrategy(Schema)  
           └─ 아니면 안전하게 → ToolStrategy(Schema)  
  

## 학습 목표

- `@tool` 데코레이터로 도구를 만들고 스키마를 확인합니다
- Pydantic 모델을 사용하여 복잡한 입력 스키마를 정의합니다
- `create_agent()`에 도구를 연결하여 에이전트를 구성합니다
- `ToolRuntime`을 통해 도구에서 런타임 컨텍스트에 접근합니다
- `with_structured_output()`으로 구조화된 출력을 설정합니다
- `ToolStrategy`와 `ProviderStrategy`의 차이를 이해합니다

## 4.1 환경 설정

API 키를 로드하고 OpenAI 모델을 초기화합니다.

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

# OpenAI를 통한 모델 초기화
model = ChatOpenAI(
    model="gpt-5.4",
)

print("모델 초기화 완료:", model.model_name)

모델 초기화 완료: gpt-5.4


In [3]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


LangSmith tracing ON — project: day1-labs-test


## 4.2 @tool 데코레이터 기본

함수에 `@tool`을 붙이면 에이전트가 사용할 수 있는 도구가 됩니다.  
LangChain은 함수의 이름, docstring, 타입 힌트를 자동으로 파싱하여 도구 스키마를 생성합니다.

```python
from langchain.tools import tool

@tool
def my_tool(param: str) -> str:
    """Tool description for the LLM."""
    return result
```

In [4]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    weather_data = {
        "Seoul": "맑음, 15\u00b0C",
        "Tokyo": "흐림, 12\u00b0C",
        "New York": "비, 8\u00b0C",
    }
    return weather_data.get(city, f"날씨 데이터를 사용할 수 없습니다: {city}")

# 도구의 스키마 확인
print("도구 이름:", get_weather.name)
print("도구 설명:", get_weather.description)
print("입력 스키마:", get_weather.args_schema.model_json_schema())

도구 이름: get_weather
도구 설명: 도시의 현재 날씨를 조회합니다.
입력 스키마: {'description': '도시의 현재 날씨를 조회합니다.', 'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_weather', 'type': 'object'}


## 4.3 Pydantic 복잡한 스키마

더 복잡한 입력 구조가 필요한 경우, Pydantic `BaseModel`을 사용하여 스키마를 정의합니다.  
`@tool(args_schema=MySchema)` 형태로 전달하면, LLM이 정확한 파라미터 구조를 이해할 수 있습니다.

- `Field(description=...)`: 각 필드에 대한 설명을 LLM에 전달
- `Field(default=...)`: 기본값 설정

In [5]:
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    """데이터베이스 쿼리용 검색 파라미터입니다."""
    query: str = Field(description="검색 쿼리 문자열")
    max_results: int = Field(default=5, description="반환할 최대 결과 수")
    category: str = Field(default="all", description="검색 카테고리: all, tech, science, news")

@tool(args_schema=SearchQuery)
def search_database(query: str, max_results: int = 5, category: str = "all") -> str:
    """고급 필터링 옵션으로 데이터베이스를 검색합니다."""
    return f"'{category}' 카테고리에서 '{query}'에 대한 {max_results}개의 결과를 찾았습니다"

print("복합 스키마:", search_database.args_schema.model_json_schema())

복합 스키마: {'description': '데이터베이스 쿼리용 검색 파라미터입니다.', 'properties': {'query': {'description': '검색 쿼리 문자열', 'title': 'Query', 'type': 'string'}, 'max_results': {'default': 5, 'description': '반환할 최대 결과 수', 'title': 'Max Results', 'type': 'integer'}, 'category': {'default': 'all', 'description': '검색 카테고리: all, tech, science, news', 'title': 'Category', 'type': 'string'}}, 'required': ['query'], 'title': 'SearchQuery', 'type': 'object'}


## 4.4 도구를 에이전트에 연결

`create_agent()`에 도구 리스트를 전달하면, 에이전트가 상황에 맞는 도구를 자동으로 선택하여 실행합니다.

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool1, tool2],
    system_prompt="...",
)
```

> **참고:** LangChain v1에서는 `create_react_agent`가 제거되었습니다. 반드시 `create_agent`를 사용하세요.

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[get_weather, search_database],
    system_prompt="당신은 날씨와 검색 도구를 사용할 수 있는 어시스턴트입니다.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨가 어떤가요?"}]},
    config=lf_config,
)
print("에이전트 응답:", result["messages"][-1].content)

에이전트 응답: 죄송하지만 지금은 서울의 실시간 날씨 데이터를 가져오지 못했습니다.

원하시면 제가 대신:
- 서울의 일반적인 현재 계절 날씨 경향을 알려드리거나
- 다시 조회를 시도할 수 있도록 도시명을 영어로 바꿔 확인해볼 수 있어요.


## 4.5 ToolRuntime

`ToolRuntime`을 사용하면 도구 함수 내에서 현재 대화 상태(state)에 접근할 수 있습니다.  
이를 통해 메시지 이력, 설정값 등 런타임 컨텍스트를 활용하는 도구를 만들 수 있습니다.

```python
@tool
def my_tool(runtime: ToolRuntime) -> str:
    messages = runtime.state["messages"]
    # ...
```

In [7]:
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(runtime: ToolRuntime) -> str:
    """현재 대화에 대한 정보를 가져옵니다."""
    messages = runtime.state["messages"]
    return f"현재 대화에 {len(messages)}개의 메시지가 있습니다."

agent_with_runtime = create_agent(
    model=model,
    tools=[get_user_info],
    system_prompt="get_user_info 도구를 사용하여 대화 정보를 확인할 수 있습니다.",
)

result = agent_with_runtime.invoke(
    {"messages": [{"role": "user", "content": "우리 대화에 메시지가 몇 개 있나요?"}]},
    config=lf_config,
)
print("응답:", result["messages"][-1].content)

응답: 현재 대화에는 2개의 메시지가 있습니다.


In [8]:
from langchain.tools import tool, ToolRuntime
import json

@tool
def inspect_runtime(runtime: ToolRuntime) -> str:
    """ToolRuntime 객체의 내부를 모두 출력합니다 (디버그용)."""
    print("=" * 60)
    print("[ToolRuntime 내부 들여다보기]")
    print("=" * 60)

    # 1) 타입과 최상위 속성
    print(f"\n▶ 타입: {type(runtime).__name__}")
    print(f"▶ dir(runtime) (밑줄 제외):")
    for attr in dir(runtime):
        if not attr.startswith("_"):
            print(f"   - {attr}")

    # 2) state — 그래프 전체 상태 (messages 등)
    print("\n▶ runtime.state 키 목록:")
    state = runtime.state
    for k in state.keys():
        v = state[k]
        preview = f"list(len={len(v)})" if isinstance(v, list) else str(v)[:80]
        print(f"   - {k}: {preview}")

    # 2-1) messages 한 줄씩 미리보기
    print("\n▶ runtime.state['messages'] 상세:")
    for i, m in enumerate(state.get("messages", [])):
        msg_type = type(m).__name__
        content = getattr(m, "content", "")
        content_preview = (content[:60] + "...") if isinstance(content, str) and len(content) > 60 else content
        print(f"   [{i}] {msg_type}: {content_preview}")

    # 3) tool_call_id — 지금 이 도구 호출의 고유 ID
    print(f"\n▶ runtime.tool_call_id: {getattr(runtime, 'tool_call_id', '(없음)')}")

    # 4) context — invoke 시 넘긴 사용자 컨텍스트 (있으면)
    print(f"\n▶ runtime.context: {getattr(runtime, 'context', '(없음)')}")

    # 5) store — 장기 메모리 (있으면)
    print(f"\n▶ runtime.store: {getattr(runtime, 'store', '(없음)')}")

    print("=" * 60)
    return f"runtime 내부 출력 완료 (state 키 {len(state)}개, 메시지 {len(state.get('messages', []))}개)"

inspect_agent = create_agent(
    model=model,
    tools=[inspect_runtime],
    system_prompt="사용자가 무엇을 묻든 반드시 inspect_runtime 도구를 한 번 호출한 뒤 답하세요.",
)

result = inspect_agent.invoke(
    {"messages": [{"role": "user", "content": "런타임 내부 좀 보여줘."}]},
    config=lf_config,
)
print("\n[최종 응답]", result["messages"][-1].content)


[ToolRuntime 내부 들여다보기]

▶ 타입: ToolRuntime
▶ dir(runtime) (밑줄 제외):
   - config
   - context
   - state
   - store
   - stream_writer
   - tool_call_id

▶ runtime.state 키 목록:
   - messages: list(len=2)

▶ runtime.state['messages'] 상세:
   [0] HumanMessage: 런타임 내부 좀 보여줘.
   [1] AIMessage: 

▶ runtime.tool_call_id: call_kxK0TgGJMeR1fkJ4DOFmjel1

▶ runtime.context: None

▶ runtime.store: None

[최종 응답] inspect_runtime 결과: runtime 내부 출력 완료 (state 키 1개, 메시지 2개)


State 1개 안에 [0],[1] 2개씩.

## 4.6 구조화된 출력

`with_structured_output()`을 사용하면 모델의 응답을 Pydantic 모델이나 dataclass 형태로 직접 받을 수 있습니다.  
이 방법은 에이전트 없이 모델에서 직접 사용합니다.

```python
structured_model = model.with_structured_output(MySchema)
result = structured_model.invoke("...")
# result는 MySchema 인스턴스
```

In [9]:
# 방법 1: with_structured_output() -- 모델 직접 사용
from pydantic import BaseModel

class MovieReview(BaseModel):
    """구조화된 영화 리뷰."""
    title: str
    rating: float
    summary: str
    recommended: bool

structured_model = model.with_structured_output(MovieReview)

review = structured_model.invoke("크리스토퍼 놀란 감독의 영화 '인셉션'을 리뷰해주세요.", config=lf_config)
print(f"제목: {review.title}")
print(f"평점: {review.rating}")
print(f"요약: {review.summary}")
print(f"추천: {'예' if review.recommended else '아니오'}")

제목: 인셉션
평점: 9.2
요약: 크리스토퍼 놀란의 '인셉션'은 꿈속의 꿈이라는 독창적인 설정을 바탕으로, 현실과 무의식의 경계를 치밀하게 설계한 SF 스릴러입니다. 복잡한 구조와 철학적 주제를 다루면서도 긴장감 있는 액션과 강렬한 비주얼을 놓치지 않습니다. 레오나르도 디카프리오를 비롯한 배우들의 몰입감 있는 연기, 한스 짐머의 압도적인 음악, 그리고 마지막까지 여운을 남기는 결말이 특히 인상적입니다. 다만 서사가 다소 복잡해 처음 감상할 때는 따라가기 어렵게 느껴질 수 있습니다.
추천: 예


도구가 필요 없으면 with_structured_output이 정답입니다. 에이전트 형태가 꼭 필요하거나 도구를 끼워야 한다면 create_agent(..., response_format=ToolStrategy(MovieReview))로 바꾸면 결과가 result["structured_response"]로 떨어집니다.

## 4.7 ToolStrategy

에이전트에서 구조화된 출력을 사용하는 방법입니다.

| 전략 | 설명 | 장점 |
|------|------|------|
| `ToolStrategy` | 도구 호출 메커니즘을 활용하여 구조화된 출력 생성 | 모든 모델에서 동작, 안정적 |

In [10]:
# 방법 2: ToolStrategy로 에이전트 감싸기
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel

class MovieReview(BaseModel):
    """구조화된 영화 리뷰."""
    title: str
    rating: float
    summary: str
    recommended: bool

review_agent = create_agent(
    model=model,
    tools=[],    # 지금은 도구없음
    system_prompt=(
        "당신은 영화 평론가입니다. "
        "사용자가 요청한 영화에 대해 제목, 10점 만점 평점, "
        "한 단락 요약, 추천 여부를 작성하세요."
    ),
    response_format=ToolStrategy(MovieReview),
)

result = review_agent.invoke(
    {"messages": [{"role": "user", "content": "크리스토퍼 놀란 감독의 영화 '인셉션'을 리뷰해주세요."}]},
    config=lf_config,
)
review = result["structured_response"]

print(f"제목: {review.title}")
print(f"평점: {review.rating}")
print(f"요약: {review.summary}")
print(f"추천: {'예' if review.recommended else '아니오'}")


제목: 인셉션
평점: 9.1
요약: 크리스토퍼 놀란의 '인셉션'은 꿈속의 꿈이라는 다층 구조를 활용해 현실과 무의식의 경계를 집요하게 파고드는 SF 스릴러입니다. 레오나르도 디카프리오가 연기한 코브의 죄책감과 상실감은 정교한 설정과 거대한 액션 스펙터클 속에서도 영화의 감정적 중심을 단단히 붙잡고 있으며, 한스 짐머의 음악과 놀란 특유의 치밀한 편집은 긴장감을 극대화합니다. 다만 복잡한 규칙과 빠른 정보 제시는 일부 관객에게 차갑고 어렵게 느껴질 수 있지만, 그 퍼즐을 따라가는 과정 자체가 이 작품의 가장 큰 매력입니다.
추천: 예


In [ ]:
# 방법 2: ToolStrategy + 도구 끼우기 (날씨영화평론가)
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel

class MovieReview(BaseModel):
    """구조화된 영화 리뷰."""
    title: str
    rating: float
    summary: str
    recommended: bool

review_agent = create_agent(
    model=model,
    tools=[get_weather],   # ← 위에서 정의한 get_weather 끼우기
    system_prompt=(
        "당신은 날씨와 영화를 관련지어 평가하는 영화 평론가입니다. "
        "리뷰 작성 전에 반드시 get_weather('Seoul') 도구를 호출해 "
        "오늘 날씨를 확인하고, 그 날씨에 어울리는 영화인지 함께 평가하세요. "
        "최종 응답에는 제목, 10점 만점 평점, 한 단락 요약(날씨 어울림 언급 포함), 추천 여부를 작성하세요."
    ),
    response_format=ToolStrategy(MovieReview),
)

result = review_agent.invoke(
    {"messages": [{"role": "user", "content": "크리스토퍼 놀란 감독의 영화 '인셉션'을 리뷰해주세요."}]},
    config=lf_config,
)
review = result["structured_response"]

print(f"제목: {review.title}")
print(f"평점: {review.rating}")
print(f"요약: {review.summary}")
print(f"추천: {'예' if review.recommended else '아니오'}")


제목: 인셉션
평점: 9.0
요약: 서울의 오늘 날씨는 맑고 15°C로, 공기가 선명해 생각이 또렷해지는 날이다. 그런 점에서 《인셉션》은 의외로 잘 어울린다. 영화는 꿈과 현실의 경계를 층층이 쌓아 올리며 복잡한 규칙을 제시하지만, 크리스토퍼 놀란의 연출은 이를 압도적인 비주얼과 긴장감 있는 리듬으로 끝까지 밀어붙인다. 레오나르도 디카프리오의 감정선은 차가운 설정 속에 인간적인 무게를 더하고, 한스 짐머의 음악은 시간의 팽창과 불안을 체감하게 만든다. 맑은 날의 선명한 집중력으로 따라가기에 특히 좋은 작품이지만, 동시에 끝나고 나서도 여운이 길게 남는 철학적 질문까지 품고 있다.
추천: 예


사용자: "인셉션 리뷰해줘"  
   ↓  
LLM: get_weather("Seoul") 호출    ← 시스템 프롬프트가 시켰으니  
   ↓  
도구: "맑음, 15°C"  
   ↓  
LLM: 영화 + 날씨 결합해서 MovieReview 객체 생성  
   ↓  
review.summary 안에 "맑은 가을 날씨에 잘 어울리는..." 같은 멘트 포함  


## 4.8 요약

이 노트북에서 학습한 핵심 내용을 정리합니다.

| 항목 | 설명 |
|------|------|
| `@tool` 데코레이터 | 함수를 에이전트용 도구로 변환 |
| `args_schema` | Pydantic 모델로 복잡한 입력 스키마 정의 |
| `create_agent()` | 모델과 도구를 연결하여 에이전트 생성 |
| `ToolRuntime` | 도구 내에서 런타임 상태(대화 이력 등) 접근 |
| `with_structured_output()` | 모델 응답을 Pydantic/dataclass로 구조화 |
| `ToolStrategy` | 도구 호출 방식의 구조화된 에이전트 출력 |

### 다음 단계
→ **[05_memory_and_streaming.ipynb](./05_memory_and_streaming.ipynb)**: 메모리와 스트리밍을 배웁니다.
